TODO:

1. Confirm all E2E scans are exported
2. Load all E2E scans
3. Run preprocessing on each scan
   - extract volume
   - extract RPE/BM
   - flatten using RPE
   - extract below-RPE ROI

4. Compute QC table
   - missing layers
   - failed flattening
   - ROI dimensions
   - intensity summaries/histograms
   - scan quality
   - acquisition parameters

5. Save processed outputs
   - flattened volume or ROI volume
   - metadata/QC CSV

6. Create clinician review file
   - ID
   - representative B-scans to review
   - barcode status: absent/present/unsure
   - optional notes

7. After clinician labeling, merge labels with QC table
   - id
   - barcode status
   - number of B-scans
   - flatten ok
   - ROI extracted
   - dimensions

Final File Structure:
```
id, barcode_status, label_confidence, notes, n_bscans, missing_layers, flatten_ok, roi_extracted, roi_shape, scan_quality
```

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from barcode.data_loading import load_e2e
from barcode.preprocessing import preprocess_volume_from_layers

DATA_DIR = PROJECT_ROOT / "data" / "heyex"

patient_dirs = sorted([p for p in DATA_DIR.iterdir() if p.is_dir()])
print("Patients found:", len(patient_dirs))
print(patient_dirs[:5])

In [ ]:
def get_e2e_paths(data_dir, n_patients=None):
    patient_dirs = sorted([p for p in Path(data_dir).iterdir() if p.is_dir()])

    if n_patients is not None:
        patient_dirs = patient_dirs[:n_patients]

    rows = []

    for patient_dir in patient_dirs:
        patient_id = patient_dir.name

        e2e_files = sorted(patient_dir.glob("*.E2E")) + sorted(patient_dir.glob("*.e2e"))

        for e2e_path in e2e_files:
            rows.append({
                "patient_id": patient_id,
                "e2e_path": e2e_path,
                "file_name": e2e_path.name,
            })

    return rows

In [ ]:
rows = get_e2e_paths(DATA_DIR, n_patients=1)

print("E2E files found:", len(rows))
rows[:5]

In [ ]:
def preprocess_batch(data_dir, n_patients=1):
    rows = get_e2e_paths(data_dir, n_patients=n_patients)
    results = []

    for i, row in enumerate(rows, start=1):
        patient_id = row["patient_id"]
        e2e_path = row["e2e_path"]

        print(f"[{i}/{len(rows)}] Patient {patient_id} | {e2e_path.name}")

        result = {
            "patient_id": patient_id,
            "file_name": e2e_path.name,
            "path": str(e2e_path),
            "loaded_ok": False,
            "preprocess_ok": False,
            "error": None,
        }

        try:
            ev = load_e2e(e2e_path)

            result["loaded_ok"] = True
            result["shape"] = ev.data.shape
            result["n_bscans"] = ev.data.shape[0]
            result["height"] = ev.data.shape[1]
            result["width"] = ev.data.shape[2]
            result["available_layers"] = list(ev.layers.keys())
            result["has_rpe"] = "RPE" in ev.layers
            result["has_bm"] = "BM" in ev.layers

            if not result["has_rpe"]:
                raise ValueError("Missing RPE layer")

            out = preprocess_volume_from_layers(
                volume=ev.data,
                rpe=ev.layers["RPE"].data,
                offset_top=5,
                offset_bottom=160,
                normalization="zscore",
                normalization_mode="global",
            )

            result["preprocess_ok"] = True
            result["target_y"] = out["target_y"]
            result["flattened_shape"] = out["flattened_volume"].shape
            result["roi_shape"] = out["roi_volume"].shape
            result["processed_roi_shape"] = out["processed_roi"].shape

        except Exception as e:
            result["error"] = repr(e)

        results.append(result)

    return results

In [ ]:
results_1 = preprocess_batch(DATA_DIR, n_patients=1)
results_1

In [ ]:
results_5 = preprocess_batch(DATA_DIR, n_patients=5)
results_5

In [ ]:
import pandas as pd

qc_df = pd.DataFrame(results_5)
qc_df